# 第9回 状態空間モデル①：カルマンフィルタとベイズ更新

このノートブックは、スライドと同じ順に並んでいます。
コードはすべて完成しているので、上から順に実行して、結果を読んでください。

- 0節：セットアップ（授業の最初に実行する）
- A節：理論をシミュレーションで確かめる
- B節：実データを解析する
- C節：設定を変えて回し直す

今日の問い：観測が1つ入るたびに、見えない水準の推定を、どれだけ直せばよいか。

## 0. セットアップ

授業が始まったら、まずこのセルを実行してください。
パッケージのインストールに少し時間がかかります。終わるまで、次のセルは動きません。

In [ ]:
# OSのコードネームを確認（noble = Ubuntu 24.04 など）
system("lsb_release -cs", intern = TRUE)

options(
  repos = c(CRAN = "https://packagemanager.posit.co/cran/__linux__/noble/2026-09-01"),
  HTTPUserAgent = sprintf("R/%s R (%s)", getRversion(),
    paste(getRversion(), R.version["platform"], R.version["arch"], R.version["os"]))
)
install.packages(c("tsibble", "tsibbledata", "fable", "feasts", "KFAS"))
sessionInfo()

## A. 理論をシミュレーションで確かめる

In [ ]:
library(tsibble)
library(feasts)
library(fable)
library(ggtime)   # ACFの図（autoplot）
library(dplyr)
library(ggplot2)
suppressPackageStartupMessages(library(KFAS))   # 状態空間モデル（読み込み時の案内文を出さない）
options(repr.plot.width = 8, repr.plot.height = 4.5)
invisible(Sys.setlocale("LC_TIME", "C"))   # 図の月の名前を英語にする

### A-1. ベイズ更新を1回だけ

知りたい量（水準 $\mu$）は見えません。見えるのは、ノイズの乗った観測 $y$ だけです。

ベイズ統計では、知らない量についての知識を、確率分布で表します。

- 事前分布（prior distribution）：観測を見る前の知識。ここでは $\mu \sim N(m, P)$
- 観測のモデル：$y = \mu + \varepsilon$、$\varepsilon \sim N(0, \sigma_\varepsilon^2)$
- 事後分布（posterior distribution）：観測 $y$ を見たあとの知識

ベイズの定理（Bayes' theorem）は、事後分布が「観測のモデル × 事前分布」に比例する、という式です。
どちらも正規分布なら、事後分布も正規分布になり、平均と分散は次の式で求まります。

$$
K = \frac{P}{P + \sigma_\varepsilon^2}, \qquad
\text{事後の平均} = m + K\,(y - m), \qquad
\text{事後の分散} = (1 - K)\,P
$$

$K$ は、観測をどれだけ信じるかの重みです。カルマンゲイン（Kalman gain）と呼びます。

In [ ]:
# ベイズ更新：事前分布 N(m, P) と、観測 y（ノイズの分散 s2_eps）から、事後分布を出す
bayes_update <- function(m, P, y, s2_eps) {
  K <- P / (P + s2_eps)                # 観測をどれだけ信じるか（カルマンゲイン）
  list(K = K,
       mean = m + K * (y - m),         # 事後の平均：事前の平均を、外れの K 倍だけ直す
       var  = (1 - K) * P)             # 事後の分散：事前の分散の (1 - K) 倍
}
# 事前分布、観測、事後分布を描く
plot_update <- function(m, P, y, s2_eps) {
  post <- bayes_update(m, P, y, s2_eps)
  half <- 4 * sqrt(max(P, s2_eps))
  x <- seq(min(m, y) - half, max(m, y) + half, length.out = 400)
  d <- rbind(
    data.frame(x = x, curve = "1 prior", density = dnorm(x, m, sqrt(P))),
    data.frame(x = x, curve = "2 observation model", density = dnorm(y, x, sqrt(s2_eps))),
    data.frame(x = x, curve = "3 posterior", density = dnorm(x, post$mean, sqrt(post$var))))
  ggplot(d, aes(x, density, colour = curve)) +
    geom_line(linewidth = 0.8) +
    geom_vline(xintercept = y, linetype = "dashed") +
    labs(x = "Level mu", y = "Density (dashed line: observation y)", colour = NULL)
}

事前分布を $N(50, 4)$、観測ノイズの分散を4とします。観測が $y = 56$ だったとき、事後分布はどうなるでしょうか。

In [ ]:
plot_update(m = 50, P = 4, y = 56, s2_eps = 4)
unlist(bayes_update(m = 50, P = 4, y = 56, s2_eps = 4))

- 事前の不確かさと、観測の不確かさが同じなので、$K = 0.5$ です。事後の平均は、事前の平均（50）と観測（56）のまん中になります。
- 事後の分散は2で、事前の分散の半分です。山が細く、高くなります。

事前の分散 $P$ を変えると、$K$ が変わります。

In [ ]:
cases <- data.frame(P = c(0.25, 4, 64))
cases <- cbind(cases, t(sapply(cases$P, function(P) unlist(bayes_update(50, P, 56, 4)))))
round(cases, 3)

- $P$ が小さい（事前の知識に自信がある）：$K$ は0に近く、観測を見てもほとんど動きません。
- $P$ が大きい（事前の知識があいまい）：$K$ は1に近く、ほぼ観測の値になります。

式が正しいことを、シミュレーションで確かめます。
事前分布から水準を100万個引き、それぞれにノイズを足して観測を作ります。
そのうち、観測が56の近く（55.9〜56.1）だった場合だけを集めます。集めた水準の平均と分散が、事後分布の平均と分散にあたります。

In [ ]:
set.seed(1)
mu_draw <- rnorm(1e6, mean = 50, sd = 2)          # 事前分布 N(50, 4) から、水準を引く
y_draw  <- mu_draw + rnorm(1e6, mean = 0, sd = 2) # 観測 = 水準 + ノイズ
near <- abs(y_draw - 56) < 0.1                    # 観測が56の近くだった場合
round(c(n = sum(near), mean = mean(mu_draw[near]), var = var(mu_draw[near])), 2)

式の値（平均53、分散2）と、ほぼ同じになります。

### A-2. ローカルレベルモデル

第5回と同じ人工の系列を作ります。ゆっくり動く水準に、ノイズが乗っています。

$$
\begin{aligned}
y_t &= \mu_t + \varepsilon_t, & \varepsilon_t &\sim N(0, \sigma_\varepsilon^2) \\
\mu_t &= \mu_{t-1} + \eta_t, & \eta_t &\sim N(0, \sigma_\eta^2)
\end{aligned}
$$

これを、ローカルレベルモデル（local level model）と呼びます。
見えない量 $\mu_t$ を状態（state）と呼び、状態の動き方と、状態から観測が出るしくみを2本の式で書いたモデルを、状態空間モデル（state space model）と呼びます。

In [ ]:
set.seed(1)
n_sim <- 200
s2_eta <- 0.25                                         # 水準のノイズの分散（標準偏差0.5）
s2_eps <- 4                                            # 観測のノイズの分散（標準偏差2）
mu <- 50 + cumsum(rnorm(n_sim, sd = sqrt(s2_eta)))     # 水準：ランダムウォーク
y  <- mu + rnorm(n_sim, sd = sqrt(s2_eps))             # 観測：水準 ＋ ノイズ
sim <- tsibble(t = 1:n_sim, y = y, mu = mu, index = t)
ggplot(sim, aes(t)) +
  geom_line(aes(y = y), colour = "grey50") +
  geom_line(aes(y = mu), colour = "red") +
  labs(x = "Time t (step)", y = "Value (grey: observed y, red: level mu)")

見えるのは灰色の線だけです。赤い線（水準）を、灰色の線から推定します。

### A-3. カルマンフィルタ

カルマンフィルタ（Kalman filter）は、次の2つの手順を、1時点ずつくり返します。

- 予測（prediction）：前の時点の事後分布から、今の時点の事前分布を作る。水準は $\sigma_\eta^2$ の分だけ動くので、平均はそのまま、分散が $\sigma_\eta^2$ だけ広がる。
- 更新（update）：今の時点の観測 $y_t$ で、A-1のベイズ更新をする。

時点 $s$ までの観測を使った、$\mu_t$ の分布の平均を $m_{t|s}$、分散を $P_{t|s}$ と書きます。

$$
\begin{aligned}
\text{予測：} \quad & m_{t|t-1} = m_{t-1|t-1}, & P_{t|t-1} &= P_{t-1|t-1} + \sigma_\eta^2 \\
\text{更新：} \quad & m_{t|t} = m_{t|t-1} + K_t\,(y_t - m_{t|t-1}), & P_{t|t} &= (1 - K_t)\,P_{t|t-1}, \qquad K_t = \frac{P_{t|t-1}}{P_{t|t-1} + \sigma_\varepsilon^2}
\end{aligned}
$$

観測がない時点（`NA`）では、更新を飛ばして、予測だけを進めます。

In [ ]:
# ローカルレベルモデルのカルマンフィルタ
# y：観測、s2_eps：観測のノイズの分散、s2_eta：水準のノイズの分散、N(m0, P0)：最初の時点の事前分布
kalman_filter <- function(y, s2_eps, s2_eta, m0, P0) {
  n <- length(y)
  m_pred <- P_pred <- K <- m_filt <- P_filt <- numeric(n)
  m <- m0; P <- P0
  for (t in 1:n) {
    if (t > 1) P <- P + s2_eta                           # 予測：分散が s2_eta だけ広がる
    m_pred[t] <- m; P_pred[t] <- P                       # 事前分布（y[t] を見る前）
    K[t] <- if (is.na(y[t])) 0 else P / (P + s2_eps)     # 観測がなければ、更新しない
    if (!is.na(y[t])) m <- m + K[t] * (y[t] - m)         # 更新：外れの K 倍だけ直す
    P <- (1 - K[t]) * P
    m_filt[t] <- m; P_filt[t] <- P                       # 事後分布（y[t] を見たあと）
  }
  data.frame(t = 1:n, y = y, m_pred, P_pred, K, m_filt, P_filt)
}
# 定常状態のカルマンゲイン（q は分散の比 s2_eta / s2_eps）
steady_gain <- function(q) (-q + sqrt(q^2 + 4 * q)) / 2
# 誤差
rmse <- function(estimate, truth) sqrt(mean((estimate - truth)^2))

In [ ]:
# 水準の推定値（青い線）と、95%の幅（青い帯）を描く。灰色が観測、赤が本当の水準（分かるときだけ）
plot_level <- function(t, y, m, P, truth = NULL) {
  d <- data.frame(t = t, y = y, m = m, lower = m - 1.96 * sqrt(P), upper = m + 1.96 * sqrt(P))
  g <- ggplot(d, aes(t)) +
    geom_line(aes(y = y), colour = "grey60", na.rm = TRUE) +
    geom_ribbon(aes(ymin = lower, ymax = upper), fill = "blue", alpha = 0.2) +
    geom_line(aes(y = m), colour = "blue")
  if (!is.null(truth)) g <- g + geom_line(data = data.frame(t = t, truth = truth), aes(y = truth), colour = "red")
  g
}

最初の時点の事前分布は、$N(0, 100^2)$ にします。水準について、ほとんど何も知らない、という意味です。
分散は、系列を作るときに使った値（$\sigma_\varepsilon^2 = 4$、$\sigma_\eta^2 = 0.25$）を使います。

In [ ]:
kf <- kalman_filter(sim$y, s2_eps, s2_eta, m0 = 0, P0 = 100^2)
round(head(kf, 4), 3)
plot_level(kf$t, kf$y, kf$m_filt, kf$P_filt, truth = sim$mu) +
  labs(x = "Time t (step)", y = "Value (blue: filtered level, red: true level)")

- `m_pred`、`P_pred`：$y_t$ を見る前の、事前分布の平均と分散
- `m_filt`、`P_filt`：$y_t$ を見たあとの、事後分布の平均と分散。フィルタ（filtering）の結果と呼ぶ
- 1時点目は、事前分布があいまいなので、`K`はほぼ1です。水準の推定値は、ほぼ観測の値になります。

カルマンゲイン $K_t$ の動きを見ます。

In [ ]:
ggplot(kf |> filter(t <= 30), aes(t, K)) +
  geom_line() + geom_point() +
  geom_hline(yintercept = steady_gain(s2_eta / s2_eps), colour = "red", linetype = "dashed") +
  labs(x = "Time t (step)", y = "Kalman gain K_t (red: steady state)")
round(c(K_last = kf$K[n_sim], K_steady = steady_gain(s2_eta / s2_eps),
        P_filt_last = kf$P_filt[n_sim]), 4)

$K_t$ は、10時点ほどで一定の値に落ち着きます。この状態を定常状態（steady state）と呼びます。
落ち着く先は、分散の比 $q = \sigma_\eta^2 / \sigma_\varepsilon^2$ だけで決まります。

$$
K = \frac{-q + \sqrt{q^2 + 4q}}{2}
$$

これは、第5回で見た「いちばん良い $\alpha$」の式と同じです。
$K$ が一定なら、更新の式は $m_{t|t} = m_{t-1|t-1} + K\,(y_t - m_{t-1|t-1})$ となり、単純指数平滑法 $\ell_t = \ell_{t-1} + \alpha\,(y_t - \ell_{t-1})$ と同じ形になります。

確かめます。$\alpha$ を定常状態の $K$ にした単純指数平滑法と、カルマンフィルタを比べます。

In [ ]:
alpha <- steady_gain(s2_eta / s2_eps)
level <- numeric(n_sim)
level[1] <- sim$y[1]                                    # 最初の水準は、最初の観測にする
for (t in 2:n_sim) level[t] <- level[t - 1] + alpha * (sim$y[t] - level[t - 1])
compare <- data.frame(t = 1:n_sim, kalman = kf$m_filt, ses = level)
round(compare[c(1, 2, 3, 5, 10, 20, 50, 200), ], 3)

最初の数時点は少し違いますが、すぐに一致します。カルマンフィルタは、最初のうち $K_t$ が大きいからです。

最初の事前分布を変えると、どうなるでしょうか。3通りで比べます。

In [ ]:
priors <- data.frame(m0 = c(0, 50, 30), P0 = c(100^2, 1, 1),
                     label = c("1 vague: N(0, 100^2)", "2 right and sure: N(50, 1)", "3 wrong and sure: N(30, 1)"))
by_prior <- do.call(rbind, lapply(1:3, function(i)
  cbind(kalman_filter(sim$y, s2_eps, s2_eta, priors$m0[i], priors$P0[i]), prior = priors$label[i])))
ggplot(by_prior |> filter(t <= 40), aes(t, m_filt, colour = prior)) +
  geom_line() +
  geom_line(data = sim |> filter(t <= 40), aes(t, mu), colour = "black", linetype = "dashed") +
  labs(x = "Time t (step)", y = "Filtered level (dashed: true level)", colour = "Prior at t = 1")

まちがった事前分布に自信をもっていると（3）、最初のうち、推定値は事前分布に引っぱられます。
それでも、観測が増えると、3本は同じ値に近づきます。事前分布の影響は、観測が増えると消えます。

### A-4. KFASと突き合わせる

同じ計算を、`KFAS`パッケージで行います。

- `SSModel()`：モデルを作る。`SSMtrend(1, Q = ...)`が水準の式で、`Q`が $\sigma_\eta^2$、`H`が $\sigma_\varepsilon^2$。`a1`と`P1`が、最初の事前分布の平均と分散
- `KFS()`：カルマンフィルタと平滑化を実行する

In [ ]:
model <- SSModel(sim$y ~ SSMtrend(1, Q = s2_eta, a1 = 0, P1 = 100^2), H = s2_eps)
kfs <- KFS(model, filtering = "state", smoothing = "state")
c(mean_filt = max(abs(as.numeric(kfs$att) - kf$m_filt)),     # att：フィルタの平均
  var_filt  = max(abs(as.numeric(kfs$Ptt) - kf$P_filt)),     # Ptt：フィルタの分散
  mean_pred = max(abs(as.numeric(kfs$a)[1:n_sim] - kf$m_pred)))   # a：予測の平均

自分で書いたカルマンフィルタとの差は、どれもほぼ0です（計算の丸めの大きさ）。

### A-5. 平滑化

フィルタは、時点 $t$ までの観測だけで $\mu_t$ を推定します。
すべての観測 $y_1, \dots, y_n$ を使って $\mu_t$ を推定することを、平滑化（smoothing）と呼びます。あとの時点の観測も使えるので、推定が良くなります。

平滑化は、フィルタの結果を、最後の時点から前へ向かって直していきます。

$$
m_{t|n} = m_{t|t} + J_t \left( m_{t+1|n} - m_{t+1|t} \right), \qquad
P_{t|n} = P_{t|t} + J_t^2 \left( P_{t+1|n} - P_{t+1|t} \right), \qquad
J_t = \frac{P_{t|t}}{P_{t+1|t}}
$$

In [ ]:
# 平滑化：カルマンフィルタの結果 kf を、最後の時点から前へ向かって直す
kalman_smoother <- function(kf) {
  n <- nrow(kf)
  m_smooth <- kf$m_filt; P_smooth <- kf$P_filt             # 最後の時点は、フィルタと同じ
  for (t in (n - 1):1) {
    J <- kf$P_filt[t] / kf$P_pred[t + 1]
    m_smooth[t] <- kf$m_filt[t] + J * (m_smooth[t + 1] - kf$m_pred[t + 1])
    P_smooth[t] <- kf$P_filt[t] + J^2 * (P_smooth[t + 1] - kf$P_pred[t + 1])
  }
  cbind(kf, m_smooth, P_smooth)
}

In [ ]:
ks <- kalman_smoother(kf)
c(mean_smooth = max(abs(as.numeric(kfs$alphahat) - ks$m_smooth)),   # alphahat：平滑化の平均
  var_smooth  = max(abs(as.numeric(kfs$V) - ks$P_smooth)))          # V：平滑化の分散
plot_level(ks$t, ks$y, ks$m_smooth, ks$P_smooth, truth = sim$mu) +
  labs(x = "Time t (step)", y = "Value (blue: smoothed level, red: true level)")

KFASの平滑化とも一致します。フィルタの図と比べると、青い線がなめらかで、帯が細くなっています。

フィルタと平滑化を、数で比べます。

In [ ]:
round(c(rmse_filter = rmse(ks$m_filt, sim$mu), rmse_smoother = rmse(ks$m_smooth, sim$mu),
        sd_filter = sqrt(ks$P_filt[100]), sd_smoother = sqrt(ks$P_smooth[100])), 3)

- `rmse`：本当の水準との差。平滑化のほうが小さい。
- `sd`：100時点目の事後分布の標準偏差。平滑化のほうが小さい。

1本だけでは、たまたまかもしれません。同じモデルから長さ100の系列を500本作り、次の2つを調べます。

- 推定値と本当の水準の差（RMSE）は、事後分布の標準偏差と合っているか
- 95%の幅（平均 ± 1.96 × 標準偏差）は、本当の水準を95%の割合で含むか

In [ ]:
set.seed(1)
one <- function() {
  mu <- 50 + cumsum(rnorm(100, sd = sqrt(s2_eta)))
  y  <- mu + rnorm(100, sd = sqrt(s2_eps))
  ks <- kalman_smoother(kalman_filter(y, s2_eps, s2_eta, m0 = 0, P0 = 100^2))
  c(rmse_filter    = rmse(ks$m_filt, mu),
    rmse_smoother  = rmse(ks$m_smooth, mu),
    cover_filter   = mean(abs(ks$m_filt - mu) < 1.96 * sqrt(ks$P_filt)),
    cover_smoother = mean(abs(ks$m_smooth - mu) < 1.96 * sqrt(ks$P_smooth)))
}
rep_result <- rowMeans(replicate(500, one()))
ks100 <- kalman_smoother(kalman_filter(rep(0, 100), s2_eps, s2_eta, m0 = 0, P0 = 100^2))
round(c(rep_result, sd_filter = sqrt(mean(ks100$P_filt)), sd_smoother = sqrt(mean(ks100$P_smooth))), 3)

- RMSEは、事後分布の標準偏差（`sd_filter`、`sd_smoother`）とほぼ同じです。
- 95%の幅は、本当の水準を、ほぼ95%の割合で含みます。
- 事後分布の分散 $P$ は、観測の値を使わずに計算できます（式に $y_t$ が出てこない）。`ks100`では、観測をすべて0にして分散だけを計算しました。

### A-6. 分散を最尤法で推定する

ここまでは、分散 $\sigma_\varepsilon^2$、$\sigma_\eta^2$ に、系列を作ったときの値を使いました。実データでは、分散も推定します。

カルマンフィルタは、$y_t$ を見る前に、$y_t$ の予測分布を出しています。平均は $m_{t|t-1}$、分散は $F_t = P_{t|t-1} + \sigma_\varepsilon^2$ です。
予測の外れ $v_t = y_t - m_{t|t-1}$ を使うと、対数尤度（第3回）は次のように書けます。

$$
\log L(\sigma_\varepsilon^2, \sigma_\eta^2) = -\frac{1}{2} \sum_{t=1}^{n} \left\{ \log (2\pi F_t) + \frac{v_t^2}{F_t} \right\}
$$

これが最も大きくなる分散を選びます。分散は正の数なので、対数をとった値を動かします。

In [ ]:
# 対数尤度：1時点先の予測の外れ v と、その分散 f_var から計算する
loglik_ll <- function(kf, s2_eps) {
  v <- kf$y - kf$m_pred
  f_var <- kf$P_pred + s2_eps
  -0.5 * sum(log(2 * pi * f_var) + v^2 / f_var, na.rm = TRUE)
}
# KFASで、2つの分散を最尤法で推定する
fit_ll <- function(y, m0, P0) {
  model <- SSModel(y ~ SSMtrend(1, Q = NA, a1 = m0, P1 = P0), H = NA)   # NA が、推定する分散
  fit <- fitSSM(model, inits = log(c(var(y), var(y)) / 2), method = "BFGS")
  list(model = fit$model, s2_eps = fit$model$H[1], s2_eta = fit$model$Q[1])
}

自分で書いた対数尤度を`optim()`で最大にした結果と、KFASの`fitSSM()`の結果を比べます。

In [ ]:
neg_loglik <- function(par) {                       # par は、2つの分散の対数
  kf <- kalman_filter(sim$y, exp(par[1]), exp(par[2]), m0 = 0, P0 = 100^2)
  -loglik_ll(kf, exp(par[1]))
}
opt <- optim(log(c(var(sim$y), var(sim$y)) / 2), neg_loglik, method = "BFGS")
fit_sim <- fit_ll(sim$y, m0 = 0, P0 = 100^2)
mle <- rbind(true = c(s2_eps, s2_eta),
             mine = exp(opt$par),
             KFAS = c(fit_sim$s2_eps, fit_sim$s2_eta))
colnames(mle) <- c("s2_eps", "s2_eta")
mle <- cbind(mle, q = mle[, "s2_eta"] / mle[, "s2_eps"])
cbind(mle, K_steady = steady_gain(mle[, "q"])) |> round(3)
c(loglik_mine = -opt$value, loglik_KFAS = as.numeric(logLik(fit_sim$model)))

- 2つの推定値は一致します。対数尤度も同じです。
- 本当の値（4と0.25）とは、少しずれます。200個のデータから推定しているからです。

### A-7. 予測と、その幅

系列の先を予測するときは、観測がないので、更新ができません。予測の手順だけを、くり返します。
$h$ 時点先の観測 $y_{n+h}$ の予測分布は、次のようになります。

$$
\text{平均} = m_{n|n}, \qquad \text{分散} = P_{n|n} + h\,\sigma_\eta^2 + \sigma_\varepsilon^2
$$

自分で書いたカルマンフィルタは、観測が`NA`の時点で更新を飛ばします。系列のうしろに`NA`を10個つなげば、10時点先までの予測になります。

In [ ]:
h_sim <- 10
kf_fc <- kalman_filter(c(sim$y, rep(NA, h_sim)), fit_sim$s2_eps, fit_sim$s2_eta, m0 = 0, P0 = 100^2)
mine <- tail(kf_fc, h_sim)
pred <- predict(fit_sim$model, n.ahead = h_sim, interval = "prediction", level = 0.95)
data.frame(step = 1:h_sim,
           mean_mine = mine$m_pred,
           sd_mine   = sqrt(mine$P_pred + fit_sim$s2_eps),            # 観測の予測なので、観測のノイズを足す
           mean_KFAS = as.numeric(pred[, "fit"]),
           sd_KFAS   = as.numeric(pred[, "upr"] - pred[, "fit"]) / qnorm(0.975)) |> round(3)
plot_level(kf_fc$t, kf_fc$y, kf_fc$m_pred, kf_fc$P_pred + fit_sim$s2_eps) +
  coord_cartesian(xlim = c(150, n_sim + h_sim)) +
  labs(x = "Time t (step)", y = "Value (blue: one-step and multi-step prediction of y)")

- 予測の平均は、最後の時点の水準の推定値のまま、水平です。
- 予測の幅は、先へ行くほど広がります。水準が、これから動くからです。
- 系列の途中に`NA`があっても、同じことが起きます。欠測（第2回）の扱いは、第10回で取り上げます。

### A-8. 同じモデルの、3つの顔

ローカルレベルモデルは、これまでの回のモデルと、次のように対応します。

- 第5回のETS(A,N,N)（単純指数平滑法）：$\alpha$ が、定常状態のカルマンゲイン $K$ にあたる
- 第4回のARIMA(0,1,1)：MAの係数 $\theta$ が、$K - 1$ にあたる

3つを同じ系列に当てはめて、確かめます。

In [ ]:
fit3 <- sim |>
  model(ets   = ETS(y ~ error("A") + trend("N") + season("N")),
        arima = ARIMA(y ~ 0 + pdq(0, 1, 1)))
est3 <- tidy(fit3)
gain3 <- c(local_level = steady_gain(fit_sim$s2_eta / fit_sim$s2_eps),
           ets_alpha   = est3$estimate[est3$term == "alpha"],
           arima_1_plus_theta = 1 + est3$estimate[est3$term == "ma1"])
round(gain3, 3)
fc3 <- fit3 |> forecast(h = 1)
round(c(local_level = as.numeric(pred[1, "fit"]),
        ets   = fc3$.mean[fc3$.model == "ets"],
        arima = fc3$.mean[fc3$.model == "arima"]), 3)

- 1行目：観測をどれだけ信じるかの重み。3つとも、ほぼ同じです。
- 2行目：1時点先の予測。3つとも、ほぼ同じです。
- 小さな違いは、最初の時点の扱い（最初の水準を推定するか、事前分布を置くか）から来ます。

## B. 実データを解析する

Rに付属する`Nile`を使います。エジプトのアスワンで測った、ナイル川の年ごとの流量です。

B節は、A節で作った関数（`kalman_filter`、`kalman_smoother`、`steady_gain`、`rmse`、`plot_level`、`fit_ll`）を使います。A節を実行してから進んでください。

### B-0. 設定

このセルの値を変えると、B節の解析がすべて変わります。まずはこのまま実行してください。

In [ ]:
# ここを変えると、以下の解析がすべて変わる
year_from  <- 1871    # 始まりの年：1871〜1900
year_to    <- 1970    # 終わりの年：1950〜1970
h          <- 10      # テストデータの年数：5 か 10
prior_mean <- 1000    # 最初の年の事前分布の平均：0〜2000
prior_sd   <- 1000    # 最初の年の事前分布の標準偏差：30〜1000（小さいほど、事前分布を強く信じる）
q_my       <- 1       # 自分で決める分散の比 q：0.001〜10（大きいほど、観測を強く信じる）

### B-1. データを確かめる

`Nile`は、Rの古い形式（`ts`）の時系列です。`tsibble`に直してから確かめます。

In [ ]:
class(Nile)
c(start = start(Nile)[1], end = end(Nile)[1], frequency = frequency(Nile))
nile_all <- as_tsibble(Nile) |>
  rename(year = index, flow = value)
glimpse(nile_all)
interval(nile_all)
c(rows = nrow(nile_all), missing = sum(is.na(nile_all$flow)), gaps = has_gaps(nile_all)$.gaps)

- 1行は1年です。`flow`が年間の流量（$10^8\,\mathrm{m}^3$）です。
- 期間は1871年から1970年、100行です。
- 欠測（`NA`）も、行の抜けもありません。

### B-2. 前処理

期間を絞り、最後の`h`年をテストデータにします。年ごとの値なので、季節はありません。

In [ ]:
nile <- nile_all |>
  filter(year >= year_from, year <= year_to) |>
  mutate(y = flow)
n <- nrow(nile)
train <- nile |> filter(year <= year_to - h)
test  <- nile |> filter(year >  year_to - h)
m0 <- prior_mean       # 最初の年の事前分布の平均
P0 <- prior_sd^2       # 最初の年の事前分布の分散
c(n = n, train = nrow(train), test = nrow(test))

In [ ]:
ggplot(nile, aes(year, y)) +
  annotate("rect", xmin = year_to - h + 0.5, xmax = year_to + 0.5, ymin = -Inf, ymax = Inf,
           alpha = 0.15, fill = "red") +
  geom_line() +
  labs(x = "Year", y = "Annual flow of the Nile (10^8 m^3)")

赤い帯がテストデータです。

### B-3. 分散を推定する

訓練データにローカルレベルモデルを当てはめ、2つの分散を最尤法で推定します。

In [ ]:
fit <- fit_ll(train$y, m0, P0)
q_hat <- fit$s2_eta / fit$s2_eps
est <- c(s2_eps = fit$s2_eps, s2_eta = fit$s2_eta, q = q_hat, K_steady = steady_gain(q_hat))
round(est, 4)

- `s2_eps`：観測のノイズの分散。`s2_eta`：水準のノイズの分散。
- `q`：分散の比。`K_steady`：定常状態のカルマンゲイン。新しい観測を、この割合だけ信じます。

第5回の単純指数平滑法（ETS(A,N,N)）を当てはめて、$\alpha$ と比べます。

In [ ]:
alpha_ets <- train |>
  model(ets = ETS(y ~ error("A") + trend("N") + season("N"))) |>
  tidy() |>
  filter(term == "alpha") |>
  pull(estimate)
round(c(K_steady = steady_gain(q_hat), alpha_ets = alpha_ets), 3)

### B-4. フィルタと平滑化

In [ ]:
kfs <- KFS(fit$model, filtering = "state", smoothing = "state")
state <- data.frame(year = train$year, y = train$y,
                    m_filt   = as.numeric(kfs$att),      P_filt   = as.numeric(kfs$Ptt),
                    m_smooth = as.numeric(kfs$alphahat), P_smooth = as.numeric(kfs$V))
ks_my <- kalman_smoother(kalman_filter(train$y, fit$s2_eps, fit$s2_eta, m0, P0))
c(filter   = max(abs(ks_my$m_filt - state$m_filt)),      # 自分で書いた関数との差
  smoother = max(abs(ks_my$m_smooth - state$m_smooth)))

自分で書いたカルマンフィルタと平滑化は、実データでもKFASと一致します。

In [ ]:
plot_level(state$year, state$y, state$m_filt, state$P_filt) +
  labs(x = "Year", y = "Annual flow (blue: filtered level)")

In [ ]:
plot_level(state$year, state$y, state$m_smooth, state$P_smooth) +
  labs(x = "Year", y = "Annual flow (blue: smoothed level)")

フィルタは、その年までのデータで水準を推定します。平滑化は、訓練データのすべての年を使います。
2つの図で、水準が下がり始める年と、帯の幅を比べてください。

### B-5. 診断

モデルが合っていれば、1時点先の予測の外れを標準偏差で割った $e_t = v_t / \sqrt{F_t}$ は、標準正規分布に従うホワイトノイズになります。

In [ ]:
resid <- tsibble(year = train$year, e = as.numeric(rstandard(kfs, "recursive")), index = year)
ggplot(resid, aes(year, e)) +
  geom_col() +
  geom_hline(yintercept = c(-1.96, 1.96), colour = "blue", linetype = "dashed") +
  labs(x = "Year", y = "Standardized one-step prediction error")
worst <- resid |> as_tibble() |> slice_max(abs(e), n = 3)
worst

In [ ]:
resid |>
  ACF(e, lag_max = 20) |>
  autoplot() +
  labs(x = "Lag h (year)", y = "Sample ACF of standardized errors")
lb <- resid |> features(e, ljung_box, lag = 10)
lb

- 棒が青い破線（±1.96）の外に出る年が、全体の5%くらいなら、仮定どおりです。とびぬけて大きい年がないかを見ます。
- ACFが破線の中にあり、リュング・ボックス検定（第4回）のp値が小さくなければ、外れに従属性は残っていません。

### B-6. 分散の比を、自分で決める

設定セルの`q_my`を分散の比にして、自分で書いたカルマンフィルタを動かします。
観測のノイズの分散は推定した値のままにし、水準のノイズの分散を`q_my`倍にします。

In [ ]:
kf_my <- kalman_filter(train$y, fit$s2_eps, q_my * fit$s2_eps, m0, P0)
round(c(K_ml = steady_gain(q_hat), K_my = steady_gain(q_my)), 3)
both <- rbind(data.frame(year = train$year, model = "ll (maximum likelihood)", level = state$m_filt),
              data.frame(year = train$year, model = "ll_my (q = q_my)", level = kf_my$m_filt))
ggplot(both, aes(year, level, colour = model)) +
  geom_line(data = train, aes(year, y), colour = "grey60") +
  geom_line() +
  labs(x = "Year", y = "Annual flow and filtered level", colour = "Model")

`q_my`が大きいほど、水準の推定値は観測を追いかけます。小さいほど、動かなくなります。

### B-7. 予測を比べる

訓練データで当てはめ、テストデータの`h`年を予測します。比べるのは次の5つです。

- `mean`：平均法、`naive`：ナイーブ法（第2回のベンチマーク。年ごとの値で季節がないので、季節ナイーブ法は使いません）
- `ets`：単純指数平滑法（第5回）
- `ll`：ローカルレベルモデル（分散は最尤推定）
- `ll_my`：ローカルレベルモデル（分散の比は`q_my`）

In [ ]:
# ローカルレベルモデルで、h 時点先まで予測する
# ll：分散は最尤推定（lower、upper は95%の予測区間）、ll_my：分散の比を q_my に決めたもの
forecast_ll <- function(y, h, m0, P0, q_my) {
  fit <- fit_ll(y, m0, P0)
  pred <- predict(fit$model, n.ahead = h, interval = "prediction", level = 0.95)
  kf_my <- kalman_filter(y, fit$s2_eps, q_my * fit$s2_eps, m0, P0)
  data.frame(step = 1:h,
             ll = as.numeric(pred[, "fit"]),
             lower = as.numeric(pred[, "lwr"]), upper = as.numeric(pred[, "upr"]),
             ll_my = kf_my$m_filt[length(y)])
}
# 誤差を1行にまとめる
acc_row <- function(name, pred, truth) {
  data.frame(.model = name, RMSE = rmse(pred, truth), MAE = mean(abs(pred - truth)))
}

In [ ]:
fc <- train |>
  model(mean  = MEAN(y),
        naive = NAIVE(y),
        ets   = ETS(y ~ error("A") + trend("N") + season("N"))) |>
  forecast(h = h)
fc_ll <- cbind(year = test$year, forecast_ll(train$y, h, m0, P0, q_my))
fc_all <- rbind(as.data.frame(fc)[, c("year", ".model", ".mean")],
                data.frame(year = fc_ll$year, .model = "ll", .mean = fc_ll$ll),
                data.frame(year = fc_ll$year, .model = "ll_my", .mean = fc_ll$ll_my))
ggplot(nile |> filter(year > year_to - 4 * h), aes(year)) +
  geom_ribbon(data = fc_ll, aes(ymin = lower, ymax = upper), fill = "blue", alpha = 0.15) +
  geom_line(aes(y = y)) +
  geom_line(data = fc_all, aes(y = .mean, colour = .model)) +
  labs(x = "Year", y = "Annual flow (band: 95% prediction interval of ll)", colour = "Method")

In [ ]:
acc <- rbind(as.data.frame(accuracy(fc, nile))[, c(".model", "RMSE", "MAE")],
             acc_row("ll", fc_ll$ll, test$y),
             acc_row("ll_my", fc_ll$ll_my, test$y))
acc
cover <- mean(test$y >= fc_ll$lower & test$y <= fc_ll$upper)   # 95%の予測区間に入った割合
cover

### B-8. 時系列交差検証

訓練データを`h`年ずつ伸ばしながら、同じ評価を4回くり返します。分散は、そのたびに推定し直します。

In [ ]:
cv <- nile |>
  filter(year <= year_to - h) |>
  stretch_tsibble(.init = n - 4 * h, .step = h)
acc_cv_fable <- cv |>
  model(mean  = MEAN(y),
        naive = NAIVE(y),
        ets   = ETS(y ~ error("A") + trend("N") + season("N"))) |>
  forecast(h = h) |>
  accuracy(nile)
fc_cv <- do.call(rbind, lapply(split(as_tibble(cv), cv$.id), function(d)
  cbind(year = max(d$year) + 1:h, forecast_ll(d$y, h, m0, P0, q_my))))
truth <- nile$y[match(fc_cv$year, nile$year)]
acc_cv <- rbind(as.data.frame(acc_cv_fable)[, c(".model", "RMSE", "MAE")],
                acc_row("ll", fc_cv$ll, truth),
                acc_row("ll_my", fc_cv$ll_my, truth))
acc_cv

読み方：

- B-3：`K_steady`と`alpha_ets`は、ほぼ同じになるはずです。ローカルレベルモデルと単純指数平滑法は、同じモデルの2つの書き方だからです。
- B-4：ローカルレベルモデルは、水準が毎年少しずつ動く、と考えます。水準が1年で大きく変わったとき、フィルタは何年か遅れて追いつきます。
- B-5：とびぬけて大きい外れがあれば、その年に、正規分布のノイズでは説明しにくいことが起きています。
- B-7、B-8：`ll`と`ets`は、ほぼ同じ予測になります。ベンチマークに勝つかどうかは、期間によって変わります。1回の評価で優劣を決めないでください。

## C. 設定を変えて回し直す

B-0の設定セルを書き替えて、B節を上からすべて実行し直してください
（Colabでは、設定セルを選んで、メニューの［ランタイム］→［以降のセルを実行］）。

変えられる値：

| 変数 | 値 |
| --- | --- |
| `year_from` | 1871〜1900 |
| `year_to` | 1950〜1970 |
| `h` | 5、10 |
| `prior_mean` | 0〜2000 |
| `prior_sd` | 30〜1000 |
| `q_my` | 0.001〜10 |

読むときの観点：

1. `year_from`を1899にすると、推定した`s2_eta`と`K_steady`はどう変わるか。そのとき、`ll`の予測は、どのベンチマークと同じになるか。
2. `q_my`を0.001、0.1、10と変えると、B-6の水準の推定値はどう変わるか。`ll_my`の予測は、どのベンチマークに近づくか。
3. `prior_mean`を500、`prior_sd`を30にすると（まちがった事前分布を、強く信じる）、B-4のフィルタの最初の数年と、B-3の分散の推定値はどう変わるか。
4. `prior_sd`を30のまま、`prior_mean`を1100にすると（ほぼ正しい事前分布を、強く信じる）、結果は最初の設定とどのくらい違うか。
5. `year_to`や`h`を変えると、B-7とB-8で、方法の順位は変わるか。

正解は1つではありません。自分の設定で出た結果を、最初の設定（1871〜1970年、10年先、事前分布 $N(1000, 1000^2)$、`q_my = 1`）の結果と比べてください。